# Producer 3

In [3]:
from kafka import KafkaProducer
import json
import random
import time
from datetime import datetime

In [4]:
# Подключаемся к Kafka
producer = KafkaProducer(
    bootstrap_servers="localhost:9092",

    # Преобразуем Python-словарь в JSON
    value_serializer=lambda x: json.dumps(x).encode("utf-8"),

    # Ключ сообщения — ID клиента
    key_serializer=lambda x: str(x).encode("utf-8")
)

In [5]:
# Создаем словарик с ID продукта и ценами
products = {
    301: 1500,
    302: 2500,
    303: 5000,
    304: 7500
}

In [6]:
print("Producer started")
# Симуляция создания заказа в интернет-магазине
try:
    for i in range(10):
        # Генерируем случайный ID клиента 
        customer_id = random.randint(1, 5)
        # Выбираем случайный продукт из словаря
        product_id = random.choice(list(products.keys()))
        
        event = {
            "order_id": i + 1,
            "customer_id": customer_id,
            "product_id": product_id,
            "amount": products[product_id],
            "timestamp": datetime.now().isoformat()
        }

        # Отправляем событие в Kafka
        future = producer.send(
            "sales",
            key=customer_id,
            value=event
        )

        # Ждем подтверждения от Kafka
        metadata = future.get(timeout=10)

        print(
            f"Sent: {event} | "
            f"partition={metadata.partition}, "
            f"offset={metadata.offset}"
        )

        time.sleep(3)

finally:
    producer.flush()
    producer.close()

Producer started
Sent: {'order_id': 1, 'customer_id': 1, 'product_id': 301, 'amount': 1500, 'timestamp': '2026-11-02T20:12:54.792661'} | partition=0, offset=0
Sent: {'order_id': 2, 'customer_id': 3, 'product_id': 301, 'amount': 1500, 'timestamp': '2026-11-02T20:12:58.299311'} | partition=0, offset=1
Sent: {'order_id': 3, 'customer_id': 4, 'product_id': 301, 'amount': 1500, 'timestamp': '2026-11-02T20:13:01.315145'} | partition=0, offset=2
Sent: {'order_id': 4, 'customer_id': 1, 'product_id': 302, 'amount': 2500, 'timestamp': '2026-11-02T20:13:04.335833'} | partition=0, offset=3
Sent: {'order_id': 5, 'customer_id': 5, 'product_id': 304, 'amount': 7500, 'timestamp': '2026-11-02T20:13:07.350616'} | partition=0, offset=4
Sent: {'order_id': 6, 'customer_id': 4, 'product_id': 302, 'amount': 2500, 'timestamp': '2026-11-02T20:13:10.368681'} | partition=0, offset=5
Sent: {'order_id': 7, 'customer_id': 5, 'product_id': 304, 'amount': 7500, 'timestamp': '2026-11-02T20:13:13.388341'} | partition=0